In [ ]:
# Set PYTHONPATH to include HPGD directory
import sys
import os

# Add configurable_mdp directory to Python path
module_path = os.path.join(os.getcwd(), "configurable_mdp")  # Set PATH/TO/configurable_mdp

if module_path not in sys.path:
    sys.path.append(module_path)

# Also set environment variable for consistency
os.environ['PYTHONPATH'] = os.environ.get('PYTHONPATH', '') + f":{module_path}"

print(f"Added {module_path} to Python path")
print(f"Current PYTHONPATH: {os.environ.get('PYTHONPATH', 'Not set')}")
print(f"Current sys.path includes configurable_mdp: {module_path in sys.path}")

In [ ]:
import jax
import jax.numpy as jnp
import yaml
import pickle
import pandas as pd
from notebooks.visualization_functions import plot_UL_rewards
import matplotlib.pyplot as plt

def flatten_keys(d):
    if isinstance(d, dict):
        for k, v in d.items():
            if isinstance(v, dict):
                for sub_k in flatten_keys(v):
                    yield k+"."+sub_k
            else:
                yield k

colors = plt.cm.Dark2.colors  # 8 distinct colors
additional_colors = plt.cm.Set1.colors  # 9 additional colors
colors += additional_colors  # Extend color palette
linewidth = 2.5
alpha = 0.8
figsize = (8.0, 4.8)  # slightly smaller plot box to keep text readable after placement
base_font_size = 20
legend_font_size = 18  # keep legend relatively smaller than other text
axis_label_font_size = 22  # make axis labels slightly larger
plt.rcParams.update(
    {
        'font.size': base_font_size,
        'axes.labelsize': axis_label_font_size,
        'legend.fontsize': legend_font_size,
        'text.usetex': True,
        'axes.linewidth': linewidth,
        'xtick.major.width': linewidth,
        'ytick.major.width': linewidth,
        'xtick.major.size': 2*linewidth,
        'ytick.major.size': 2*linewidth,
        'axes.prop_cycle': plt.cycler(color=colors),
        'lines.linewidth': linewidth,
    }
)

In [ ]:
save_fig = False

data_dir = "data/experiment_bilevel_lqr_steps_10000_reg_lambda_0_1_env_3_2_SR_1_0_EC_0_5_IC_0_1_AC_0_1_itr_2e3"
save_dir = os.path.join(data_dir, "figures")

# Algorithms to plot (Note: the order in the legend follows the order in this list)
algorithms = ["proposal", "baseline", "hpgd", "hpgd_td"]

# Metrics to plot
metric_names = [
    "return_UL",
    "return_LL",
    "insulation_level",
    "airflow_adjustment",
    "LQR_total_iterations",
    "return_UL_stability",
    "return_UL_energy_cost",
    "return_UL_insulation_cost",
    "return_UL_airflow_cost",
]

# Plotting configurations
statistic = "mean"  # "mean", "median"
rolling_window = 50  # Rolling mean smoothing
max_iter = 2000     # For plotting, only show results up to this iteration
separated_in_seeds = False  # Whether the data is separated in different seeds
remove_outlier = True  # Whether to remove outlier data
remove_invalid = False  # Whether to remove invalid data (e.g., lower-level failure cases)

y_label_map = {
    "return_UL": r"Upper-level objective",
    "return_LL": r"Lower-level objective",
    "insulation_level": r"$\alpha$",
    "airflow_adjustment": r"$\beta$",
    "LQR_total_iterations": r"LQR iterations",
    "return_UL_stability": r"Stability reward",
    "return_UL_energy_cost": r"Energy cost",
    "return_UL_insulation_cost": r"Insulation cost",
    "return_UL_airflow_cost": r"Airflow cost",
}
algo_labels = {
    "baseline": r"\\textsc{Naive-PGD}",
    "hpgd": r"\\textsc{HPGD (MC)}",
    "hpgd_td": r"\\textsc{HPGD (TD)}",
    "proposal": r"\\textsc{BC-HG}",
}
line_styles = {
    "baseline": ":",
    "hpgd": "--",
    "hpgd_td": "-.",
    "proposal": "-",
}
algo_colors = {
    "baseline": colors[0],
    "hpgd": colors[1],
    "hpgd_td": colors[3],
    "proposal": colors[2],
}
zorder = {
    "baseline": 1,
    "hpgd": 2,
    "hpgd_td": 3,
    "proposal": 4,
}
algorithms = [algo for algo in algorithms if algo in algo_labels]
algo_labels = {algo: algo_labels[algo] for algo in algorithms}  # sort according to "algorithms"

# Load Data

In [ ]:
# Read config
configs = {}
dataframes = {}
run_metrics = {}
param_indices = {}
stable_masks = {}
max_masks = {}
min_masks = {}
found = []
with open(f"{data_dir}/config.yaml", "r") as f:
    config = yaml.safe_load(f)
print("Config: ", config)
rng = jax.random.PRNGKey(config["random_seed"])
for algorithm in algorithms:
    try:
        with open(f"{data_dir}/metrics_{algorithm}.pkl", "rb") as f:
            metrics = pickle.load(f)
            metrics = jax.tree_map(lambda x: x.astype(jnp.float32), metrics)
        with open(f"{data_dir}/update_dict_{algorithm}.pkl", "rb") as f:
            update_dict = pickle.load(f)
    except FileNotFoundError:
        continue
    print("Algorithm: ", algorithm)
    found.append(algorithm)

    return_UL = metrics["return_UL"] if len(update_dict) > 0 else metrics["return_UL"][:, None, :]  # (n_seeds, n_parameters, n_steps)
    max_indices = jnp.argmax(return_UL, axis=0)  # (n_parameters, n_steps)
    min_indices = jnp.argmin(return_UL, axis=0)  # (n_parameters, n_steps)
    max_mask = jnp.zeros_like(return_UL, dtype=bool)  # (n_seeds, n_parameters, n_steps)
    min_mask = jnp.zeros_like(return_UL, dtype=bool)  # (n_seeds, n_parameters, n_steps)
    for i in range(return_UL.shape[1]): # n_parameters
        max_mask = max_mask.at[max_indices[i], i, jnp.arange(return_UL.shape[2])].set(True)
        min_mask = min_mask.at[min_indices[i], i, jnp.arange(return_UL.shape[2])].set(True)
    if not len(update_dict) > 0:
        max_mask, min_mask = max_mask[:, 0, :], min_mask[:, 0, :]

    if len(update_dict) > 0:
        update_dict_names = list(flatten_keys(update_dict))
        indices = jax.tree_util.tree_leaves(update_dict, is_leaf=lambda x: isinstance(x, list))
        indices = [x.tolist() for x in indices]
        indices = [[str(x) if isinstance(x, list) else x for x in l] for l in indices]
        multi_index = pd.MultiIndex.from_arrays(
            indices,
            names=update_dict_names
        )
        return_UL = (metrics["return_UL"] if not remove_outlier 
                     else jnp.where(max_mask | min_mask, jnp.nan, metrics["return_UL"]))
        df = pd.DataFrame(
            {
                "returns": jnp.nanmean(jnp.nanmean(return_UL[...,-1:], -1), 0),
                "std": jnp.nanstd(jnp.nanmean(return_UL[...,-1:], -1), 0),
                "returns_start": jnp.nanmean(jnp.nanmean(return_UL[...,:10], -1), 0),
                "returns_end": jnp.nanmean(jnp.nanmean(return_UL[...,-10:], -1), 0),
                "index": jnp.arange(multi_index.shape[0])
            },
            index=multi_index
        )
        df = df.sort_values("returns_end", ascending=False)
    else:
        return_UL = (metrics["return_UL"] if not remove_outlier 
                    else jnp.where(max_mask | min_mask, jnp.nan, metrics["return_UL"]))
        df = pd.DataFrame(
            {
                "returns": jnp.mean(jnp.mean(return_UL[...,-1:], -1), 0),
                "std": jnp.nanstd(jnp.mean(return_UL[...,-1:], -1), 0),
                "index": jnp.arange(1)
            },
            index=pd.Index(["all"])
        )
    print("Grid search parameters:", update_dict)
    display(df)
    
    dataframes[algorithm] = df
    run_metrics[algorithm] = metrics
    configs[algorithm] = config
    param_indices[algorithm] = df["index"].iloc[0].item()  # parameter with the higher return improvement ratio
    if "LQR_total_iterations" in metrics:
        stable_masks[algorithm] = (
            metrics["LQR_total_iterations"] < config["lower_optimisation"]["training"]["max_steps"]  # (n_seeds, n_parameters, n_steps)
        )
    max_masks[algorithm] = max_mask
    min_masks[algorithm] = min_mask
print(f"Found algorithms: {found}")

In [ ]:
# NaN check
for algo in found:
    for metric in run_metrics[algo]:
        nan_exist = jnp.any(jnp.isnan(run_metrics[algo][metric]))
        if nan_exist:
            print(f"NaN exist in {metric} for {algo}")

In [ ]:
# Select parameters with the best performance improvement for each algorithm
_max_iter = min(max_iter, min([run_metrics[alg]["return_UL"].shape[-1] for alg in run_metrics.keys()]))
if len(update_dict) > 0:
    print("Best parameter indices: ", param_indices)
    metrics_for_plot = {
        alg: {metric_name: run_metrics[alg][metric_name][
                :, param_indices[alg], :_max_iter, ...    # (n_seeds, n_parameters, n_steps, ...)
            ] for metric_name in run_metrics[alg]}
        for alg in run_metrics.keys()
    }
    stable_masks_for_plot, max_masks_for_plot, min_masks_for_plot = {}, {}, {}
    _for_plot = [stable_masks_for_plot, max_masks_for_plot, min_masks_for_plot]
    _l = [stable_masks, max_masks, min_masks]
    for i in range(len(_for_plot)):
        for alg in _l[i].keys():
            _for_plot[i][alg] = _l[i][alg][
                :, param_indices[alg], :_max_iter    # (n_seeds, n_parameters, n_steps)
            ]

    metrics_for_plot_same_param = [
        {alg: {metric_name: run_metrics[alg][metric_name][
                :, i, :_max_iter, ...    # (n_seeds, n_parameters, n_steps, ...)
            ] for metric_name in run_metrics[alg]}
            for alg in run_metrics.keys()
        } for i in range(len(multi_index))
    ]
else:
    metrics_for_plot = {
        alg: {metric_name: run_metrics[alg][metric_name][
                :, :_max_iter, ...    # (n_seeds, n_steps, ...)
            ] for metric_name in run_metrics[alg]}
        for alg in run_metrics.keys()
    }
    stable_masks_for_plot, max_masks_for_plot, min_masks_for_plot = {}, {}, {}
    _for_plot = [stable_masks_for_plot, max_masks_for_plot, min_masks_for_plot]
    _l = [stable_masks, max_masks, min_masks]
    for i in range(len(_for_plot)):
        for alg in _l[i].keys():
            _for_plot[i][alg] = _l[i][alg][
                :, :_max_iter    # (n_seeds, n_steps)
            ]

# Plot Results

In [1]:
if not os.path.exists(save_dir) and save_fig:

    os.makedirs(save_dir)

for metric_name in metric_names:
    print(f"Plotting metric: {metric_name}")
    y_data = {}
    legend_names = {}
    _line_styles = {}
    _algo_colors = {}
    _zorder = {}
    for j, algorithm in enumerate(algorithms):
        if algorithm not in metrics_for_plot:
            continue
        else:
            metrics = metrics_for_plot[algorithm]
            stable_mask = stable_masks_for_plot[algorithm]
            max_mask = max_masks_for_plot[algorithm]
            min_mask = min_masks_for_plot[algorithm]

        if metric_name not in metrics:
            print(f"Metric {metric_name} not found for algorithm {algorithm}, skipping.")
            continue

        if metric_name in ["insulation_level", "airflow_adjustment"]:
            arr = metrics[metric_name]  # (n_seeds, n_steps, dim_insulation_level)
            ylim = (0.0, 1.0)
            for i in range(arr.shape[-1]):
                name = f"{algorithm}_{metric_name}_{i+1}"
                y_data[name] = arr[...,i]
                legend_names[name] = (
                    algo_labels[algorithm] + rf" $\alpha_{i+1}$" if metric_name == "insulation_level" 
                    else algo_labels[algorithm] + rf" $\beta_{i+1}$"
                )
                ls = ["-", "--", "-.", ":"]
                _line_styles[name] = ls[i % len(ls)]
                _algo_colors[name] = algo_colors[algorithm]
                _zorder[name] = zorder[algorithm]
        else:
            arr = metrics[metric_name]  # (n_seeds, n_steps)

            if metric_name not in ["LQR_total_iterations", "LQR_positive_definit_S_rate", "LQR_controllability"]:
                if remove_invalid:
                    print(f"[{algorithm}] {jnp.sum(jnp.logical_not(stable_mask))} invalid data removed.")
                    arr = jnp.where(
                        stable_mask,
                        arr,
                        jnp.nan,
                    )  # Remove invalid data with unstable LQR parameterization

                if remove_outlier:
                    arr = jnp.where(
                        max_mask | min_mask,
                        jnp.nan,
                        arr,
                    )  # Remove outlier data
                    print(f"[{algorithm}] Outlier data removed for each time step.")

            name = algorithm
            y_data[name] = arr
            legend_names[name] = algo_labels[algorithm]
            _line_styles = line_styles
            _algo_colors = algo_colors
            _zorder = zorder
            
    ylim = None
    legend_position = None
    if metric_name in ["insulation_level", "airflow_adjustment"]:
        ylim = (0.0, 1.0)
    elif metric_name in ["return_UL"]:
        ylim = (-295, -185)
        legend_position={
            "loc": "lower center",
            "bbox_to_anchor": (0.5, 0.0),
            "ncol": 2,
        }
    elif metric_name in ["return_LL"]:
        # ylim = (-1700, -1000)
        legend_position={
            "loc": "lower center",
            "bbox_to_anchor": (0.5, 0.0),
            "ncol": 2,
        }
    file_name = metric_name
    if metric_name not in ["LQR_total_iterations", "LQR_positive_definit_S_rate", "LQR_controllability"]:
        if remove_invalid:
            file_name += "_no_invalid"
        if remove_outlier:
            file_name += "_no_outliers"

    plot_UL_rewards(
        y_data,
        rolling_window=rolling_window,
        figsize=figsize,
        legend_names=legend_names,
        legend_position=legend_position,
        line_styles=_line_styles,
        algo_colors=_algo_colors,
        zorder=_zorder,
        y_ticks=None,
        ylim=ylim,
        ylabel=y_label_map[metric_name],
        statistic=statistic,  # "mean", "median"
        alpha=alpha,
        savefig_path=os.path.join(save_dir, f"{file_name}.pdf") if save_fig else None,
        separated_in_seeds=separated_in_seeds,  # Changed: data is separated in seeds
    )

NameError: name 'os' is not defined

In [ ]:
# Plot for all parameter settings

ylims = {
    "return_UL": (-500, -180),
    "return_UL_stability": (-350, -50),
    "return_UL_energy_cost": (220, 330),
    "return_UL_insulation_cost": (0, 45),
    "return_UL_airflow_cost": (0, 45),
    "insulation_level": (0, 1),
    "airflow_adjustment": (0, 1),
}

def tuple_to_filename(params_tuple, precision=3):
    """数値tupleをファイル名用文字列に変換"""
    # 各要素を指定精度で丸めて文字列化
    rounded_params = [round(x, precision) for x in params_tuple]
    return "-".join(map(str, rounded_params)).replace(".", "_")

for metric_name in metric_names:
    # サブプロットの数を決定
    n_params = len(multi_index)
    if n_params == 0:
        continue
    
    # 横一列に配置するサブプロットを作成
    fig, axes = plt.subplots(1, n_params, figsize=(figsize[0] * n_params, figsize[1]))
    
    # 単一パラメータの場合は axes をリストに変換
    if n_params == 1:
        axes = [axes]
    
    for idx, ((_metrics_for_plot, p), ax) in enumerate(zip(zip(metrics_for_plot_same_param, multi_index), axes)):
        print(f"Plot for parameter setting {idx}: ", p)
        filename_label = tuple_to_filename(p, precision=4)
        y_data = {}
        legend_names = {}
        _line_styles = {}
        _algo_colors = {}
        _zorder = {}
        
        for j, algorithm in enumerate(algorithms):
            if algorithm not in _metrics_for_plot:
                continue
            else:
                metrics = _metrics_for_plot[algorithm]
                stable_mask = stable_masks[algorithm][:,idx,:]  # (n_seeds, n_steps)

            if metric_name not in metrics:
                print(f"Metric {metric_name} not found for algorithm {algorithm}, skipping.")
                continue
            
            if metric_name in ["insulation_level", "airflow_adjustment"]:
                arr = metrics[metric_name]  # (n_seeds, n_steps, dim_insulation_level)
                ylim = (0.0, 1.0)
                for i in range(arr.shape[-1]):
                    name = f"{algorithm}_{metric_name}_{i+1}"
                    y_data[name] = arr[...,i]
                    legend_names[name] = (
                        algo_labels[algorithm] + rf" $\alpha_{i+1}$" if metric_name == "insulation_level" 
                        else algo_labels[algorithm] + rf" $\beta_{i+1}$"
                    )
                    ls = ["-", "--", "-.", ":"]
                    _line_styles[name] = ls[i % len(ls)]
                    _algo_colors[name] = algo_colors[algorithm]
                    _zorder[name] = zorder[algorithm]
            else:
                arr = metrics[metric_name]  # (n_seeds, n_steps)

                if not metric_name in ["LQR_total_iterations", "LQR_positive_definit_S_rate", "LQR_controllability"]:
                    if remove_invalid:
                        print(f"[{algorithm}] {jnp.sum(jnp.logical_not(stable_mask))} invalid data removed.")
                        arr = jnp.where(
                            stable_mask,
                            arr,
                            jnp.nan,
                        )  # Remove invalid data with unstable LQR parameterization

                    if remove_outlier:
                        max_indices = jnp.nanargmax(arr, axis=0)  # (n_steps,)
                        min_indices = jnp.nanargmin(arr, axis=0)  # (n_steps,)
                        arr = arr.at[max_indices, jnp.arange(arr.shape[1])].set(jnp.nan)
                        arr = arr.at[min_indices, jnp.arange(arr.shape[1])].set(jnp.nan)
                        print(f"[{algorithm}] Outlier data removed for each time step.")

                name = algorithm
                y_data[name] = arr
                legend_names[name] = algo_labels[algorithm]
                _line_styles = line_styles
                _algo_colors = algo_colors
                _zorder = zorder

        # 各サブプロットにプロット
        plot_UL_rewards(
            y_data,
            rolling_window=rolling_window,
            figsize=None,  # サブプロット使用時はfigsize不要
            ax=ax,  # 特定の軸を指定
            legend_names=legend_names,
            line_styles=_line_styles,
            algo_colors=_algo_colors,
            zorder=_zorder,
            y_ticks=None,
            ylim=(0.0, 1.0) if metric_name in ["insulation_level", "airflow_adjustment"] else None,
            ylabel=y_label_map[metric_name] if idx == 0 else "",  # 左端のみY軸ラベル
            statistic=statistic,  # "mean", "median"
            alpha=alpha,
            show_legend=(idx == n_params - 1),  # 右端のみ凡例表示
            title=f"({filename_label})",  # パラメータ設定をタイトルに
            separated_in_seeds=separated_in_seeds,  # Changed: data is separated in seeds
        )

    # 全体のタイトルと調整
    p_name_label = ""
    for p in multi_index.names: 
        p_name_label += f"{p}-"
    if y_label_map[metric_name].startswith("$"):
        title = y_label_map[metric_name]
    else:
        title = f'{y_label_map[metric_name].title()}'
    title += f' - All Parameter Settings ({p_name_label[:-1]})'
    fig.suptitle(title, fontsize=24)
    plt.tight_layout()
    
    # 保存
    if save_fig:
        file_name = f"{metric_name}_all_params"
        if not metric_name in ["LQR_total_iterations", "LQR_positive_definit_S_rate", "LQR_controllability"]:
            if remove_invalid:
                file_name += "_no_invalid"
            if remove_outlier:
                file_name += "_no_outliers"
            plt.savefig(os.path.join(save_dir, f"{file_name}.pdf"), bbox_inches='tight', dpi=300)
    
    plt.show()

# Demonstrate the learned model

In [ ]:
import copy
import orbax
import os
from train_bilevel_lqr_hpgd import (
    setup_environment,
    update_transition_params,
    create_trajectory_batch_sample
)
from src.algorithms.regularized_lqr import (
    create_regularized_lqr,
    Transition,
    RegularizedLQROutputs,
    RegularizedLQRPolicy,
    create_state_value_fn,
    create_q_value_fn,
    update_dictionary,
)
from src.models.StaticModel import create_state_model as create_static_train_state
from src.train.utils import update_nested_pytree, remove_non_list_entries

In [ ]:
load_data = {}
configs = {}
update_dicts = {}
for algo in found:   

    # Load checkpoint (currently NOT used)
    try:
        # loadable only with GPU-compatible JAX 
        print("\n"+"Loading ", algo)
        ckpt_path = os.path.join(os.path.abspath(data_dir), f"checkpoint_incentive_{algo}")
        checkpointer = orbax.checkpoint.PyTreeCheckpointer()
        restored = checkpointer.restore(ckpt_path)
        load_data[algo] = restored
    except:
        print(f"Could not load checkpoint for {algo}, skipping.")

    # update_dictの各パラメータ組み合わせに対して設定を更新
    cfgs = []
    updates = []
    update_dict_leaves = jax.tree_util.tree_leaves(update_dict)
    n_configs = len(update_dict_leaves[0]) if update_dict_leaves else 1
    for i in range(n_configs):
        # 各パラメータのi番目の値を取得
        update_values = jax.tree_util.tree_map(lambda x: x[i] if isinstance(x, jnp.ndarray) else x, update_dict)
        # 設定を更新
        updated_config = update_dictionary(copy.deepcopy(config), update_values)
        cfgs.append(updated_config)
        updates.append(update_values)

        configs[algo] = cfgs
        update_dicts[algo] = updates

In [ ]:
## -- DEMONSTRATE A CERTAIN PARAMETER SETTING -- ##

train_seed_idx = 0
epoch_idx = -1  # which epoch to demonstrate the trained parameters

get_trajectory_batches = {}
env_params_dict = {}
lqr_outputs = {}
envs = {}
for algo in found:
    print("\n"+f"Restore {algo}" + f" (epoch {epoch_idx})".replace("-1", "final"))
    # upper_level_outputs = load_data[algo]['upper_level_train_states']
    # lower_level_outputs = load_data[algo]['lower_level_train_states']
    param_idx = param_indices[algo]

    # Restore the environment
    cfg = configs[algo][param_idx]
    env, env_params = setup_environment(cfg["environment"])

    # Restore upper-level training states with the trained parameters
    trained_params = {
        'insulation_level': metrics_for_plot[algo]['insulation_level'][
            train_seed_idx, epoch_idx, :
        ],
        'airflow_adjustment': metrics_for_plot[algo]['airflow_adjustment'][
            train_seed_idx, epoch_idx, :
        ],
    }
    upper_level_states = {
        "insulation_level": create_static_train_state(
            param_shape=(env_params.transition_params.num_zones,),
            init_value=trained_params['insulation_level'],
            **cfg["upper_optimisation"]["model_params"],
        ),
        "airflow_adjustment": create_static_train_state(
            param_shape=(env_params.transition_params.num_zones,),
            init_value=trained_params['airflow_adjustment'],
            **cfg["upper_optimisation"]["model_params"],
        ),
    }
    env_params = update_transition_params(
        env_params, upper_level_states
    )

    # Initialize lower-level training states
    lower_level_states = RegularizedLQROutputs(
        A=env_params.transition_params.A,
        B=env_params.transition_params.B,
        Q=env_params.reward_params.Q,
        R=env_params.reward_params.R,
        W=jnp.eye(env_params.transition_params.A.shape[0]),
        P=jnp.copy(env_params.reward_params.Q),
        v=jnp.array(0.0),
        Ks=jnp.zeros((
            env_params.transition_params.B.shape[1],  # action_dim
            env_params.transition_params.A.shape[0]   # state_dim
        )),
        Sigma=jnp.eye(env_params.transition_params.B.shape[1]),  # action_dim x action_dim
        sqrtSigma=jnp.eye(env_params.transition_params.B.shape[1])  # action_dim x action_dim
    )
    # Re-train the lower-level policy from scratch
    print(f"[{algo}] Optimizing lower-level policy...")
    regularized_lqr = create_regularized_lqr(env, cfg['lower_optimisation'])
    lower_level_states, lqr_metrics, _ = regularized_lqr(
        env_params,
        lower_level_states,
    )
    policy = RegularizedLQRPolicy(
        Ks=lower_level_states.Ks,
        Sigma=lower_level_states.Sigma,
        sqrtSigma=lower_level_states.sqrtSigma,
    )

    # Cache the results
    get_trajectory_batch = create_trajectory_batch_sample(
        {"num_envs": config["upper_optimisation"]["num_envs"], ""
        "num_eval_steps": config["upper_optimisation"]["num_eval_steps"]},
        env,
        for_eval=True,
    )
    get_trajectory_batches[algo] = get_trajectory_batch
    env_params_dict[algo] = env_params
    lqr_outputs[algo] = lower_level_states
    envs[algo] = env

    # Print trained parameters
    print(f"[{algo}] Trained Parameters:")
    print("  Hyperparameters:", update_dicts[algo][param_idx])
    print("  Training Seed Index:", train_seed_idx)
    for param_name in ['insulation_level', 'airflow_adjustment']:
        if param_name in trained_params:
            print(f"  {param_name.replace('_', ' ').title()}:", trained_params[param_name])
    print(f"[{algo}] Parameterized A:\n", env.compute_parameterized_A(env_params.transition_params))
    print(f"[{algo}] LQR Metrics:")
    print("  Total Steps:", lqr_metrics['total_steps'])
    print("  Riccati Error:", lqr_metrics['Riccati_error'])
    print("  Value Error:", lqr_metrics['value_error'])
    print("  Controllability:", lqr_metrics['controllability'])
    print("  LQR_positive_definit_S_rate:", jnp.nanmean(jnp.where(
        lqr_metrics["logdet_sign"] < 0, 0.0, lqr_metrics["logdet_sign"]
    )))

In [ ]:
## -- Rollout with trained parameters -- ##
seed = 42

return_UL_stability_dict = {}
return_UL_energy_cost_dict = {}
return_UL_insulation_cost_dict = {}
return_UL_airflow_cost_dict = {}
return_UL_dict = {}
return_LL_dict = {}
for algo in found:
    cfg = configs[algo][param_indices[algo]]
    get_trajectory_batch = get_trajectory_batches[algo]
    env_params = env_params_dict[algo]
    lqr_output = lqr_outputs[algo]
    env = envs[algo]

    key = jax.random.PRNGKey(seed)
    key, _rng = jax.random.split(key)
    traj_batch = get_trajectory_batch(
        _rng,
        lqr_output,
        env_params,
    )  # Shape: (n_steps, num_envs, PyTree Structure of Transition)
    upper_rewards, info = jax.vmap(
        jax.vmap(
            env.upper_level_reward, 
            in_axes=(0,0,None)
        ), in_axes=(0,0,None)
    )(traj_batch.state, traj_batch.action, env_params)
    discounting_arr = jnp.power(
        cfg["lower_optimisation"]["discount_factor"],
        traj_batch.state.time,
    )  # Shape: (n_steps, num_envs)
    num_episodes = jnp.clip(jnp.sum(traj_batch.done), a_min=1)
    reward_UL_stability = info["stability_reward"] * discounting_arr
    reward_UL_energy_cost = info["energy_cost"] * discounting_arr
    reward_UL_insulation_cost = info["insulation_cost"] * discounting_arr
    reward_UL_airflow_cost = info["airflow_cost"] * discounting_arr
    reward_LL = traj_batch.reward * discounting_arr

    return_UL_stability = jnp.sum(reward_UL_stability) /num_episodes
    return_UL_energy_cost = jnp.sum(reward_UL_energy_cost) / num_episodes
    return_UL_insulation_cost = jnp.sum(reward_UL_insulation_cost) / num_episodes
    return_UL_airflow_cost = jnp.sum(reward_UL_airflow_cost) / num_episodes
    return_UL = (
        env_params.reward_params.stability_weight * return_UL_stability
        - env_params.reward_params.energy_weight * return_UL_energy_cost
        - env_params.reward_params.insulation_cost_weight * return_UL_insulation_cost
        - env_params.reward_params.airflow_cost_weight * return_UL_airflow_cost
    )
    return_LL = jnp.sum(reward_LL) / num_episodes

    plt.figure(figsize=figsize)
    obs_plot = traj_batch.obs[:env_params.max_steps_in_episode, 0, :]  # (time_steps, state_dim)
    return_UL_stability_plot = jnp.sum(reward_UL_stability[:env_params.max_steps_in_episode, 0])  # scalar
    return_UL_energy_cost_plot = jnp.sum(reward_UL_energy_cost[:env_params.max_steps_in_episode, 0])  # scalar
    return_UL_insulation_cost_plot = jnp.sum(reward_UL_insulation_cost[:env_params.max_steps_in_episode, 0])  # scalar
    return_UL_airflow_cost_plot = jnp.sum(reward_UL_airflow_cost[:env_params.max_steps_in_episode, 0])  # scalar
    return_UL_plot = (
        env_params.reward_params.stability_weight * return_UL_stability_plot
        - env_params.reward_params.energy_weight * return_UL_energy_cost_plot
        - env_params.reward_params.insulation_cost_weight * return_UL_insulation_cost_plot
        - env_params.reward_params.airflow_cost_weight * return_UL_airflow_cost_plot
    )
    return_LL_plot = jnp.sum(reward_LL[:env_params.max_steps_in_episode, 0])  # scalar
    for i in range(4):
        plt.plot(obs_plot[:, i], label=f'$s_{i+1}$')

    plt.xlabel('Time Step')
    plt.ylabel('State')
    plt.title(f'State ({algo})')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.grid(True, alpha=0.3)
    plt.show()

    print(f"Rollout and visualize for {algo}")
    print(f"[{algo}] Upper-level Return:", return_UL_plot)
    print(f"  Stability Reward:", return_UL_stability_plot)
    print(f"  Energy Cost:", return_UL_energy_cost_plot)
    print(f"  Insulation Cost:", return_UL_insulation_cost_plot)
    print(f"  Airflow Cost:", return_UL_airflow_cost_plot)
    print(f"[{algo}] Lower-level Return:", return_LL_plot)
    # print(f"[{algo}] States:\n", obs)

    return_UL_stability_dict[algo] = return_UL_stability
    return_UL_energy_cost_dict[algo] = return_UL_energy_cost
    return_UL_insulation_cost_dict[algo] = return_UL_insulation_cost
    return_UL_airflow_cost_dict[algo] = return_UL_airflow_cost
    return_UL_dict[algo] = return_UL
    return_LL_dict[algo] = return_LL

# Make a table of returns
import pandas as pd
return_table = pd.DataFrame({
    f"Upper-level Return": return_UL_dict,
    f"Stability Reward (× {env_params.reward_params.stability_weight})": return_UL_stability_dict,
    f"Energy Cost (× {-env_params.reward_params.energy_weight})": return_UL_energy_cost_dict,
    f"Insulation Cost (× {-env_params.reward_params.insulation_cost_weight})": return_UL_insulation_cost_dict,
    f"Airflow Cost (× {-env_params.reward_params.airflow_cost_weight})": return_UL_airflow_cost_dict,
    "Lower-level Return": return_LL_dict,
})
print("\nAverage Returns over Trajectory Rollout:")
display(return_table)